In [ ]:
# compare_models_original_test.py
import time, torch, cv2, random, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torch.utils.data import DataLoader
from torchmetrics.detection import MeanAveragePrecision
from tqdm import tqdm

# === CONFIGURATION ===
DATA_YAML = "data.yaml"
YOLO_BEST_PT = 'runs/train/sard2_yolo11_augmented7/weights/best.pt'
MASK_RCNN_PATH = "mask_rcnn_sard2_yolo_final.pth"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES_MASKRCNN = 7  # (6 classes + background)
IMG_DIR = './dataset/images/test'
LBL_DIR = './dataset/labels/test'

# === 1. ÉVALUATION YOLO ===
print("\n🔹 Évaluation YOLO sur les images originales...")
yolo_model = YOLO(YOLO_BEST_PT)
start = time.time()
metrics_yolo = yolo_model.val(split='test', data=DATA_YAML)
time_yolo = time.time() - start

yolo_results = {
    "mAP@50-95": metrics_yolo.box.map,
    "mAP@50": metrics_yolo.box.map50,
    "Precision": np.mean(metrics_yolo.box.p),
    "Recall": np.mean(metrics_yolo.box.r),
    "Time (s)": time_yolo,
}

# === 2. ÉVALUATION MASK R-CNN ===
print("\n🔹 Évaluation Mask R-CNN sur les images originales...")
from main import SARD2YOLODataset, collate_fn  # si déjà défini dans ton projet

test_dataset = SARD2YOLODataset(IMG_DIR, LBL_DIR)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

mask_model = maskrcnn_resnet50_fpn(weights=None, num_classes=NUM_CLASSES_MASKRCNN)
mask_model.load_state_dict(torch.load(MASK_RCNN_PATH, map_location=DEVICE))
mask_model.to(DEVICE).eval()

metric = MeanAveragePrecision(iou_type="bbox").to(DEVICE)

start = time.time()
with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Mask R-CNN Eval"):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        preds = mask_model(images)
        metric.update(preds, targets)
results_mask = metric.compute()
time_mask = time.time() - start

mask_results = {
    "mAP@50-95": results_mask['map'].item(),
    "mAP@50": results_mask['map_50'].item(),
    "Precision": results_mask['map'].item(),  # approx
    "Recall": results_mask['recall_large'].item(),
    "Time (s)": time_mask,
}

# === 3. COMPARAISON ===
df = pd.DataFrame([yolo_results, mask_results], index=["YOLO", "Mask R-CNN"])
print("\n📊 Comparaison sur images originales :\n", df)

df.plot(kind="bar", figsize=(10,5), title="Comparaison YOLO vs Mask R-CNN (Images Originales)")
plt.xticks(rotation=0)
plt.ylabel("Scores / Temps")
plt.tight_layout()
plt.show()

# === 4. VISUALISATION ===
print("\n🎨 Visualisation des détections...")
from main import detect_image_yolo, detect_image_maskrcnn

sample_imgs = random.sample(glob.glob(f"{IMG_DIR}/*.jpg"), 3)
for img_path in sample_imgs:
    yolo_img = detect_image_yolo(img_path)
    mask_img = detect_image_maskrcnn(img_path)

    plt.figure(figsize=(12,6))
    plt.subplot(1,2,1)
    plt.imshow(yolo_img); plt.title("YOLO"); plt.axis('off')
    plt.subplot(1,2,2)
    plt.imshow(mask_img); plt.title("Mask R-CNN"); plt.axis('off')
    plt.suptitle(f"Comparaison : {img_path}")
    plt.tight_layout()
    plt.show()


In [ ]:
# compare_models_augmented_test.py
import time, torch, cv2, random, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torch.utils.data import DataLoader
from torchmetrics.detection import MeanAveragePrecision
from tqdm import tqdm

# === CONFIGURATION ===
DATA_YAML = "data.yaml"
YOLO_BEST_PT = 'runs/train/sard2_yolo11_augmented7/weights/best.pt'
MASK_RCNN_PATH = "mask_rcnn_sard2_yolo_final.pth"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES_MASKRCNN = 7
IMG_DIR = './dataset_augmente/images/test'
LBL_DIR = './dataset_augmente/labels/test'

# === 1. ÉVALUATION YOLO ===
print("\n🔹 Évaluation YOLO sur les images augmentées...")
yolo_model = YOLO(YOLO_BEST_PT)
start = time.time()
metrics_yolo = yolo_model.val(split='test', data=DATA_YAML)
time_yolo = time.time() - start

yolo_results = {
    "mAP@50-95": metrics_yolo.box.map,
    "mAP@50": metrics_yolo.box.map50,
    "Precision": np.mean(metrics_yolo.box.p),
    "Recall": np.mean(metrics_yolo.box.r),
    "Time (s)": time_yolo,
}

# === 2. ÉVALUATION MASK R-CNN ===
print("\n🔹 Évaluation Mask R-CNN sur les images augmentées...")
from main import SARD2YOLODataset, collate_fn

test_dataset = SARD2YOLODataset(IMG_DIR, LBL_DIR)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

mask_model = maskrcnn_resnet50_fpn(weights=None, num_classes=NUM_CLASSES_MASKRCNN)
mask_model.load_state_dict(torch.load(MASK_RCNN_PATH, map_location=DEVICE))
mask_model.to(DEVICE).eval()

metric = MeanAveragePrecision(iou_type="bbox").to(DEVICE)

start = time.time()
with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Mask R-CNN Eval"):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        preds = mask_model(images)
        metric.update(preds, targets)
results_mask = metric.compute()
time_mask = time.time() - start

mask_results = {
    "mAP@50-95": results_mask['map'].item(),
    "mAP@50": results_mask['map_50'].item(),
    "Precision": results_mask['map'].item(),
    "Recall": results_mask['recall_large'].item(),
    "Time (s)": time_mask,
}

# === 3. COMPARAISON ===
df = pd.DataFrame([yolo_results, mask_results], index=["YOLO", "Mask R-CNN"])
print("\n📊 Comparaison sur images augmentées :\n", df)

df.plot(kind="bar", figsize=(10,5), title="Comparaison YOLO vs Mask R-CNN (Images Augmentées)")
plt.xticks(rotation=0)
plt.ylabel("Scores / Temps")
plt.tight_layout()
plt.show()

# === 4. VISUALISATION ===
print("\n🎨 Visualisation des détections (images augmentées)...")
from main import detect_image_yolo, detect_image_maskrcnn

sample_imgs = random.sample(glob.glob(f"{IMG_DIR}/*.jpg"), 3)
for img_path in sample_imgs:
    yolo_img = detect_image_yolo(img_path)
    mask_img = detect_image_maskrcnn(img_path)

    plt.figure(figsize=(12,6))
    plt.subplot(1,2,1)
    plt.imshow(yolo_img); plt.title("YOLO"); plt.axis('off')
    plt.subplot(1,2,2)
    plt.imshow(mask_img); plt.title("Mask R-CNN"); plt.axis('off')
    plt.suptitle(f"Comparaison : {img_path}")
    plt.tight_layout()
    plt.show()
